In [16]:
import pandas as pd
import numpy as np
import re
import joblib
from urllib.parse import urlparse
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

df = pd.read_csv('../data/malicious_phish.csv')
df['label'] = df['type'].apply(lambda x: 0 if x == 'benign' else 1)
print(df.shape)

(651191, 3)


In [17]:
IP_PATTERN = re.compile(
    r'^(?:(?:25[0-5]|2[0-4]\d|[01]?\d\d?)\.){3}(?:25[0-5]|2[0-4]\d|[01]?\d\d?)$'
)

SHORTENER_PATTERN = re.compile(
    r'bit\.ly|goo\.gl|shorte\.st|go2l\.ink|x\.co|ow\.ly|t\.co|tinyurl|tr\.im|is\.gd|cli\.gs|'
    r'yfrog\.com|migre\.me|ff\.im|tiny\.cc|url4\.eu|twit\.ac|su\.pr|twurl\.nl|snipurl\.com|'
    r'short\.to|budurl\.com|ping\.fm|post\.ly|just\.as|bkite\.com|snipr\.com|fic\.kr|loopt\.us|'
    r'doiop\.com|short\.ie|kl\.am|wp\.me|rubyurl\.com|om\.ly|to\.ly|bit\.do|lnkd\.in|'
    r'db\.tt|qr\.ae|adf\.ly|bitly\.com|cur\.lv|ity\.im|q\.gs|po\.st|bc\.vc|twitthis\.com|'
    r'u\.to|j\.mp|buzurl\.com|cutt\.us|u\.bb|yourls\.org|prettylinkpro\.com|scrnch\.me|'
    r'filoops\.info|vzturl\.com|qr\.net|1url\.com|tweez\.me|v\.gd|link\.zip\.net'
)


def is_ip_address(domain):
    host = domain.split(':')[0]
    return int(bool(IP_PATTERN.match(host)))


def has_numeric_label(domain):
    """Vrai si un des segments du domaine est entièrement numérique,
    sans être forcément une IP complète (ex. '2.exemple.com').
    Distinct de has_ip — signal plus faible mais réel (~70% de
    corrélation avec le malveillant dans ce dataset), probablement lié
    à des sous-domaines générés automatiquement (infrastructures
    C2/spam)."""
    host = domain.split(':')[0]
    return int(any(part.isdigit() for part in host.split('.')))


def has_shortener(url):
    return int(bool(SHORTENER_PATTERN.search(url.lower())))


def extract_features_v3(url):
    try:
        if not url.startswith('http'):
            url = 'http://' + url
        parsed = urlparse(url)
        domain = parsed.netloc
        path = parsed.path
    except Exception:
        domain = ''
        path = ''

    suspicious_tlds = ['.tk', '.ml', '.ga', '.cf', '.gq', '.free.fr', '.xyz', '.top']
    suspicious_words = ['login', 'secure', 'update', 'verify', 'account',
                        'banking', 'paypal', 'amazon', 'apple', 'microsoft']

    return {
        'length': len(url),
        'num_dots': url.count('.'),
        'num_hyphens': url.count('-'),
        'num_slashes': url.count('/'),
        'num_digits': sum(c.isdigit() for c in url),
        'num_at': url.count('@'),
        'num_percent': url.count('%'),
        'num_question': url.count('?'),
        'num_equal': url.count('='),
        'num_underscore': url.count('_'),
        'has_https': int(url.startswith('https')),
        'has_ip': is_ip_address(domain),
        'has_numeric_label': has_numeric_label(domain),
        'domain_length': len(domain),
        'path_length': len(path),
        'has_suspicious_tld': int(any(url.endswith(tld) or tld + '/' in url for tld in suspicious_tlds)),
        'has_suspicious_word': int(any(word in domain.lower() for word in suspicious_words)),
        'num_subdomains': len(domain.split('.')) - 2 if len(domain.split('.')) > 2 else 0,
        'has_at_in_url': int('@' in url),
        'has_shortener': has_shortener(url),
    }


print("Fonctions définies")

Fonctions définies


In [18]:
print("Extraction des features v3...")
features = df['url'].apply(extract_features_v3)
X = pd.DataFrame(features.tolist())
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Entraînement...")
rf_model_v3 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model_v3.fit(X_train, y_train)
print("Terminé")

Extraction des features v3...
Entraînement...
Terminé


In [19]:
y_pred = rf_model_v3.predict(X_test)
print(f"Accuracy : {accuracy_score(y_test, y_pred)*100:.2f}%")
print(classification_report(y_test, y_pred, target_names=['Benign', 'Malicious']))

Accuracy : 92.01%
              precision    recall  f1-score   support

      Benign       0.93      0.95      0.94     85778
   Malicious       0.89      0.87      0.88     44461

    accuracy                           0.92    130239
   macro avg       0.91      0.91      0.91    130239
weighted avg       0.92      0.92      0.92    130239



In [20]:
joblib.dump(rf_model_v3, '../models/rf_phishing_model_v3.pkl')
print("Modèle v3 sauvegardé")

Modèle v3 sauvegardé
